# Train LayoutLMv3 for GST invoices (free Colab GPU)

1. **Runtime → Change runtime type → T4 GPU**.
2. Upload the project zip (left sidebar → Files) or clone it from GitHub.
3. Run each cell top to bottom. Total time: about 30–40 minutes.

In [ ]:
# Option A: upload gst-invoice-assistant.zip to the Files panel first, then:
!unzip -q -o gst-invoice-assistant.zip
%cd gst-invoice-assistant
# Option B: !git clone https://github.com/<you>/gst-invoice-assistant && %cd gst-invoice-assistant

In [ ]:
!pip install -q -r requirements.txt -r requirements-ml.txt
import os; os.environ['PYTHONPATH'] = '.'
os.environ['DATABASE_URL'] = 'sqlite:///:memory:'

In [ ]:
# 1,500 labelled invoices across two layouts (~5 minutes)
!python -m ml.generate_dataset --n 750 --out data/layoutlm_dataset

In [ ]:
# Fine-tune (~20-30 minutes on a T4)
!python -m ml.train_layoutlm --data data/layoutlm_dataset --out models/layoutlmv3-gst --epochs 3

In [ ]:
# Compare against the rule-based extractors on invoices the model has never seen
!python -m ml.benchmark --n 60 --model models/layoutlmv3-gst

## Bring the results back
Download these two things and put them in the same place in your project:

- `data/benchmarks/results.json` → the **Model accuracy** page will show LayoutLMv3 next to the rule-based extractors.
- `models/layoutlmv3-gst/` (zip it) → set `LAYOUTLM_MODEL_PATH=models/layoutlmv3-gst` in `.env` to use it in the pipeline.

In [ ]:
!zip -qr layoutlmv3-gst.zip models/layoutlmv3-gst
from google.colab import files
files.download('layoutlmv3-gst.zip')
files.download('data/benchmarks/results.json')